## Assignment #4

<div class="assignment-notice" style="background-color: #fff3cd; border: 1px solid #ffeeba; color: #856404; padding: 10px; border-radius: 5px;">
<h4>⚠️ Please read before editing this notebook</h4>
<p>The autograder uses <strong>cell tags</strong> (e.g., <code>q1_1</code>, <code>q2_1</code>) to find your answers. Many autograder failures come from changes to the cell structure, not the code inside.</p>
<ul>
<li><strong>Put each tagged question's answer in the cell that's already there.</strong> You can add new cells for scratch work, notes, or exploration. Anything the autograder needs to grade (code, variables, text responses to questions) must live in the provided tagged cell.</li>
<li><strong>Don't delete</strong> provided cells, even if they look empty.</li>
<li><strong>Don't rename, retag, or reorder</strong> the tagged cells.</li>
</ul>
</div>

In [166]:
from grader_shared import SubmissionWidget
SubmissionWidget()


In [167]:
# Set random seed and ensure plots are displayed in the notebook
import numpy as np
import arviz as az
import pymc as pm
np.random.seed(5)

%matplotlib inline

%config Inline.figure_format = 'retina'
az.rcParams["stats.hdi_prob"] = 0.89  # sets default credible interval used by arviz

In [168]:
# utility function for standardizing variables
def standardize(series):
    return (series - series.mean()) / series.std()

**Question 1.** In the Laffer data (*Data/Laffer.csv*, first introduced in the accompanying prediction tutorial), there is one country with a high tax revenue that is an outlier. 

In the tutorial, PSIS and WAIC were used to measure the importance of this outlier in the models that were fit. 

First, **re-create and run the models provided as part of the prediction tutorial**.

Define the linear model in the cells below.

In [169]:
import pandas as pd
import numpy as np

# read in data
ex4_df = pd.read_csv("Data/Laffer.csv")

# standardize tax rate and tax revenue
tax_rate_std = (
    ex4_df["tax_rate"] - ex4_df["tax_rate"].mean()
) / ex4_df["tax_rate"].std()

tax_revenue_std = (
    ex4_df["tax_revenue"] - ex4_df["tax_revenue"].mean()
) / ex4_df["tax_revenue"].std()

In [170]:
# define the linear model and store it in m_ex4_lin
import pymc as pm

# define the linear model and store it in m_ex4_lin
with pm.Model() as m_ex4_lin:

    # Priors
    alpha = pm.Normal("alpha", 0, 1)
    beta = pm.Normal("beta", 0, 1)
    sigma = pm.Exponential("sigma", 1)

    # Linear model
    mu = alpha + beta * tax_rate_std.to_numpy()

    # Likelihood
    revenue = pm.Normal(
        "revenue",
        mu=mu,
        sigma=sigma,
        observed=tax_revenue_std.to_numpy()
    )

    # Estimate posterior using HMC (NUTS)
    idata_ex4_lin = pm.sample(
        1000,
        tune=1000,
        idata_kwargs={"log_likelihood": True}
    )



Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Define the quadratic model in the cell below.

In [171]:

# define the quadratic model and store it in m_ex4_quad
with pm.Model() as m_ex4_quad:

    # Priors
    alpha = pm.Normal("alpha", 0, 1)
    beta1 = pm.Normal("beta1", 0, 1)
    beta2 = pm.Normal("beta2", 0, 1)
    sigma = pm.Exponential("sigma", 1)

    # Quadratic model
    mu = (
        alpha
        + beta1 * tax_rate_std.to_numpy()
        + beta2 * tax_rate_std.to_numpy()**2
    )

    # Likelihood
    revenue = pm.Normal(
        "revenue",
        mu=mu,
        sigma=sigma,
        observed=tax_revenue_std.to_numpy()
    )

    # Estimate posterior using HMC MCMC
    idata_ex4_quad = pm.sample(
        1000,
        tune=1000,
        idata_kwargs={"log_likelihood": True}
    )


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta1, beta2, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 4 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Create a new **cubic** (3rd-degree polynomial) version of the model.

In [172]:

# define the cubic model and store it in m_q1_cube
with pm.Model() as m_q1_cube:

    # Priors
    alpha = pm.Normal("alpha", 0, 1)
    beta1 = pm.Normal("beta1", 0, 1)
    beta2 = pm.Normal("beta2", 0, 1)
    beta3 = pm.Normal("beta3", 0, 1)
    sigma = pm.Exponential("sigma", 1)

    # Cubic model
    mu = (
        alpha
        + beta1 * tax_rate_std.to_numpy()
        + beta2 * tax_rate_std.to_numpy()**2
        + beta3 * tax_rate_std.to_numpy()**3
    )

    # Likelihood
    revenue = pm.Normal(
        "revenue",
        mu=mu,
        sigma=sigma,
        observed=tax_revenue_std.to_numpy()
    )

    # Estimate posterior using HMC MCMC
    idata_q1_cube = pm.sample(
        1000,
        tune=1000,
        idata_kwargs={"log_likelihood": True}
    )



Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta1, beta2, beta3, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:05&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:04&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 10 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Create a new (first-order) **linear** version of the model using robust regression with a Student’s t distribution for the data distribution. Use $\nu=2$ for your Student's t distribution. 

In [173]:

# define the robust linear model
with pm.Model() as m_q1_rob_lin:

    # Priors
    alpha = pm.Normal("alpha", 0, 1)
    beta = pm.Normal("beta", 0, 1)
    sigma = pm.Exponential("sigma", 1)

    # Linear model
    mu = alpha + beta * tax_rate_std.to_numpy()

    # Student-t likelihood
    revenue = pm.StudentT(
        "revenue",
        nu=2,
        mu=mu,
        sigma=sigma,
        observed=tax_revenue_std.to_numpy()
    )

    # Estimate posterior using MCMC
    idata_q1_rob_lin = pm.sample(
        1000,
        tune=1000,
        idata_kwargs={"log_likelihood": True}
    )



Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Compare the predictive performance of the 4 models using PSIS.

In [174]:

# Store the InferenceData objects in q1_model_dict
q1_model_dict = {
    "linear": idata_ex4_lin,
    "quadratic": idata_ex4_quad,
    "cubic": idata_q1_cube,
    "robust_linear": idata_q1_rob_lin
}

# Compare models using PSIS
q1_compare_df_psis = az.compare(
    q1_model_dict,
    ic="loo",
    scale="deviance"
)

q1_compare_df_psis


/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
robust_linear,0,73.834456,3.352065,0.000000,7.996750e-01,12.446907,0.000000,False,deviance
quadratic,1,86.879362,5.825814,13.044906,2.003250e-01,18.769614,11.546741,True,deviance
cubic,2,87.021036,6.100107,13.186581,0.000000e+00,17.727167,10.674613,True,deviance
linear,3,87.970925,5.277619,14.136469,1.110223e-16,18.510114,11.045142,True,deviance


In [175]:
# compare models using PSIS
q1_compare_df_psis = az.compare(q1_model_dict, ic="loo", scale="deviance")
q1_compare_df_psis

/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
robust_linear,0,73.834456,3.352065,0.000000,7.996750e-01,12.446907,0.000000,False,deviance
quadratic,1,86.879362,5.825814,13.044906,2.003250e-01,18.769614,11.546741,True,deviance
cubic,2,87.021036,6.100107,13.186581,0.000000e+00,17.727167,10.674613,True,deviance
linear,3,87.970925,5.277619,14.136469,1.110223e-16,18.510114,11.045142,True,deviance


Of the 4 models, which one has the best predictive performance according to PSIS? Please type the best model's name used in your `q1_model_dict` in the cell below.

**⚠️ PLEASE DO NOT DELETE THIS CELL**

robust_linear


What evidence led you to choose the model stated above? Provide an explanation for your choice in the cell below. **Limit your response to no more than 2 sentences.**

**⚠️ PLEASE DO NOT DELETE THIS CELL**

_I chose the robust linear model because it has the lowest PSIS deviance (73.82) and ranks first among all four models. It also has no PSIS warning, which suggests its predictive performance estimate is more reliable._

Quantify the influence of the outlier data point in your predictive models

In [176]:

# Access PSIS values of data points
mod_rob_lin_psis = az.loo(q1_model_dict["robust_linear"], pointwise=True, scale="deviance")
mod_lin_psis = az.loo(q1_model_dict["linear"], pointwise=True, scale="deviance")
mod_quad_psis = az.loo(q1_model_dict["quadratic"], pointwise=True, scale="deviance")
mod_cube_psis = az.loo(q1_model_dict["cubic"], pointwise=True, scale="deviance")

# Create a pandas DataFrame using the Pareto-k values from each model
q1_df = pd.DataFrame(
    {
        "psis_lin": mod_lin_psis.pareto_k.values,
        "psis_cube": mod_cube_psis.pareto_k.values,
        "psis_rob_lin": mod_rob_lin_psis.pareto_k.values,
        "psis_quad": mod_quad_psis.pareto_k.values
    }
)

q1_df.head(15)


/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.py:803: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/opt/conda/envs/fnds/lib/python3.11/site-packages/arviz/stats/stats.

,psis_lin,psis_cube,psis_rob_lin,psis_quad
0,0.296991,0.759077,0.129482,0.626011
1,0.217525,0.327409,0.118014,0.244664
2,0.312467,0.346218,0.113105,0.246650
3,-0.070792,0.098665,0.070738,0.190013
4,-0.066935,0.104531,-0.066992,0.290532
5,-0.001181,0.064986,0.015739,0.310048
6,0.095860,0.014187,0.045553,0.309433
7,0.094671,0.317317,-0.003911,0.212015
8,-0.015232,-0.010480,0.020524,0.036974
9,-0.059226,0.169890,-0.123909,0.100996


Based on these results, what impact does the outlier point appear to be having on the curved relationship between tax rate and tax revenue? **Limit your response to no more than 5 sentences.**


**⚠️ PLEASE DO NOT DELETE THIS CELL**
The outlier appears to be making the relationship between tax rate and tax revenue look more curved than it actually might be. The quadratic and cubic models have high Pareto-k values for this point, suggesting that it has a strong influence on their predictions. The robust linear model is less affected by the outlier, suggesting that some of the apparent curvature may be caused by this unusual observation rather than the overall trend in the data.

**Question 2**. Reconsider the urban fox analysis (*Data/foxes.csv*) from the previous homework assignment (HW 3). 

Create models for each combination of variables that predict body weight ($W$, `weight`) in the following cells. 

**Note**: *For this exercise, you will end up creating 7 models.*

In [177]:
# Read in data from Data/foxes.csv
q2_df = pd.read_csv("Data/foxes.csv")
q2_df.head()

,group,avgfood,groupsize,area,weight
0,1,0.37,2,1.09,5.02
1,1,0.37,2,1.09,2.84
2,2,0.53,2,2.05,5.33
3,2,0.53,2,2.05,6.07
4,3,0.49,2,2.12,5.85


In [178]:

# Standardize variables and store them in the variables below
avgfood_standardized = (q2_df["avgfood"] - q2_df["avgfood"].mean()) / q2_df["avgfood"].std()

weight_standardized = (q2_df["weight"] - q2_df["weight"].mean()) / q2_df["weight"].std()

grpsz_standardized = (q2_df["groupsize"] - q2_df["groupsize"].mean()) / q2_df["groupsize"].std()

area_standardized = (q2_df["area"] - q2_df["area"].mean()) / q2_df["area"].std()


In [179]:
# Define the model and sample from the posterior
# Define the model and sample from the posterior
with pm.Model() as q2_model_1:
    a = pm.Normal("a", 0, 0.2)
    bF = pm.Normal("bF", 0, 0.5)

    mu = a + bF * avgfood_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal(
        "W",
        mu=mu,
        sigma=sigma,
        observed=weight_standardized
    )

    q2_idata_1 = pm.sample(
        idata_kwargs={"log_likelihood": True}
    )


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bF, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [180]:
# Define the model and sample from the posterior

# Define the model and sample from the posterior
with pm.Model() as q2_model_2:
    a = pm.Normal("a", 0, 0.2)
    bG = pm.Normal("bG", 0, 0.5)

    mu = a + bG * grpsz_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal(
        "W",
        mu=mu,
        sigma=sigma,
        observed=weight_standardized
    )

    q2_idata_2 = pm.sample(
        idata_kwargs={"log_likelihood": True}
    )



Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bG, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [181]:
# Define the model and sample from the posterior
with pm.Model() as q2_model_3:
    a = pm.Normal("a", 0, 0.2)
    bA = pm.Normal("bA", 0, 0.5)

    mu = a + bA * area_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal("W", mu=mu, sigma=sigma,
                  observed=weight_standardized)

    q2_idata_3 = pm.sample(idata_kwargs={"log_likelihood": True})

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bA, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:01&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 3 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [182]:

# Define the model and sample from the posterior
with pm.Model() as q2_model_4:
    a = pm.Normal("a", 0, 0.2)
    bF = pm.Normal("bF", 0, 0.5)
    bG = pm.Normal("bG", 0, 0.5)

    mu = a + bF * avgfood_standardized + bG * grpsz_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal("W", mu=mu, sigma=sigma,
                  observed=weight_standardized)
    q2_idata_4 = pm.sample(idata_kwargs={"log_likelihood": True})

    

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bF, bG, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [183]:

# Define the model and sample from the posterior
with pm.Model() as q2_model_5:
    a = pm.Normal("a", 0, 0.2)
    bF = pm.Normal("bF", 0, 0.5)
    bA = pm.Normal("bA", 0, 0.5)

    mu = a + bF * avgfood_standardized + bA * area_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal("W", mu=mu, sigma=sigma,
                  observed=weight_standardized)

    q2_idata_5 = pm.sample(idata_kwargs={"log_likelihood": True})

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bF, bA, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [184]:

# Define the model and sample from the posterior
with pm.Model() as q2_model_6:
    a = pm.Normal("a", 0, 0.2)
    bG = pm.Normal("bG", 0, 0.5)
    bA = pm.Normal("bA", 0, 0.5)

    mu = a + bG * grpsz_standardized + bA * area_standardized

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal("W", mu=mu, sigma=sigma,
                  observed=weight_standardized)
    q2_idata_6 = pm.sample(idata_kwargs={"log_likelihood": True})

    

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bG, bA, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:02&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


In [185]:
# Define the model and sample from the posterior
with pm.Model() as q2_model_7:
    a = pm.Normal("a", 0, 0.2)
    bF = pm.Normal("bF", 0, 0.5)
    bG = pm.Normal("bG", 0, 0.5)
    bA = pm.Normal("bA", 0, 0.5)

    mu = (a
          + bF * avgfood_standardized
          + bG * grpsz_standardized
          + bA * area_standardized)

    sigma = pm.Exponential("sigma", 1)

    W = pm.Normal("W", mu=mu, sigma=sigma,
                  observed=weight_standardized)
    q2_idata_7 = pm.sample(idata_kwargs={"log_likelihood": True})

    


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [a, bF, bG, bA, sigma]


<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:04&lt;00:00... Sampling chain 0, 0 divergences]</div>

<div><progress max="2000" value="2000"></progress> 100.00% [2000/2000 00:03&lt;00:00... Sampling chain 1, 0 divergences]</div>

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 8 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Compare the models with PSIS and WAIC. 

In [ ]:
# Store the final InferenceData objects from the seven models above.
q2_model_dict = {
    "avgfood": q2_idata_1,
    "groupsize": q2_idata_2,
    "area": q2_idata_3,
    "avgfood_groupsize": q2_idata_4,
    "avgfood_area": q2_idata_5,
    "groupsize_area": q2_idata_6,
    "all_three": q2_idata_7,
}

# Check that all seven fitted models are included.
assert len(q2_model_dict) == 7
assert all(isinstance(value, az.InferenceData) for value in q2_model_dict.values())


In [ ]:
# Compare the seven models using PSIS (leave-one-out cross-validation).
q2_compare_df_psis = az.compare(q2_model_dict, ic="loo", scale="deviance")
q2_compare_df_psis


In [ ]:
# Compare the seven models using WAIC.
q2_compare_df_waic = az.compare(q2_model_dict, ic="waic", scale="deviance")
q2_compare_df_waic
